# 02 - Neptune Graph
Create a Neptune Analytics graph, define the OrgIQ schema, and load sample organizational data using openCypher MERGE.

## Step 1: Load Configuration

In [ ]:
import boto3
import json
import time
from pathlib import Path
from cleanup_utils import ResourceTracker

config = json.loads(Path("config.json").read_text())
REGION = config["REGION"]
PREFIX = config["PROJECT_PREFIX"]
ACCOUNT_ID = config["ACCOUNT_ID"]

tracker = ResourceTracker()
neptune = boto3.client("neptune-graph", region_name=REGION)

print(f"\u2713 Config loaded (region={REGION}, prefix={PREFIX})")

## Step 2: Create Neptune Analytics Graph

Neptune Analytics is openCypher-only. We provision a graph with vector search enabled for embeddings.

In [ ]:
GRAPH_NAME = f"{PREFIX}-knowledge-graph"

# Check if graph already exists
existing_graphs = neptune.list_graphs()
existing = next(
    (g for g in existing_graphs["graphSummaries"] if g["name"] == GRAPH_NAME),
    None
)

if existing:
    graph_id = existing["id"]
    print(f"\u2713 Graph already exists: {graph_id}")
else:
    print(f"\u231b Creating Neptune Analytics graph: {GRAPH_NAME}...")
    response = neptune.create_graph(
        graphName=GRAPH_NAME,
        provisionedMemory=32,
        deletionProtection=False,
        publicConnectivity=True,
        vectorSearchConfiguration={
            "dimension": config["EMBEDDING_DIMENSIONS"]
        },
        tags={"project": PREFIX}
    )
    graph_id = response["id"]
    print(f"\u2713 Graph creation initiated: {graph_id}")
    tracker.add_neptune_graph(graph_id, GRAPH_NAME)

## Step 3: Wait for Graph to Become Available

In [ ]:
print("\u231b Waiting for graph to become AVAILABLE...")
while True:
    graph_info = neptune.get_graph(graphIdentifier=graph_id)
    status = graph_info["status"]
    if status == "AVAILABLE":
        endpoint = graph_info["endpoint"]
        print(f"\u2713 Graph is AVAILABLE")
        print(f"  Endpoint: {endpoint}")
        break
    elif status in ["FAILED", "DELETING"]:
        raise RuntimeError(f"Graph entered unexpected state: {status}")
    else:
        print(f"  Status: {status}, waiting 30s...")
        time.sleep(30)

## Step 4: Define OrgIQ Schema

Our graph schema represents organizational knowledge:
- **Person** — employees with expertise scores
- **Document** — authored content (articles, PRs, docs)
- **Topic** — knowledge domains
- **Team** — organizational units
- Relationships: AUTHORED, KNOWS_ABOUT, BELONGS_TO, RELATED_TO

In [ ]:
# Neptune Analytics uses executeQuery for openCypher
def run_query(query, parameters=None):
    """Execute an openCypher query against the Neptune Analytics graph."""
    kwargs = {
        "graphIdentifier": graph_id,
        "queryString": query,
        "language": "OPEN_CYPHER"
    }
    if parameters:
        kwargs["parameters"] = parameters
    response = neptune.execute_query(**kwargs)
    return json.loads(response["payload"].read())

print("\u2713 Query helper defined")

## Step 5: Create Constraint Indexes

Create indexes to support efficient lookups on key properties.

In [ ]:
# Create indexes for efficient lookups
indexes = [
    "CREATE INDEX IF NOT EXISTS FOR (p:Person) ON (p.email)",
    "CREATE INDEX IF NOT EXISTS FOR (d:Document) ON (d.doc_id)",
    "CREATE INDEX IF NOT EXISTS FOR (t:Topic) ON (t.name)",
    "CREATE INDEX IF NOT EXISTS FOR (tm:Team) ON (tm.name)",
]

for idx_query in indexes:
    try:
        run_query(idx_query)
        print(f"  \u2713 Index created: {idx_query.split('(')[1].split(')')[0]}")
    except Exception as e:
        if "already exists" in str(e).lower():
            print(f"  \u2713 Index already exists")
        else:
            print(f"  \u2717 Index error: {e}")

print("\u2713 Indexes configured")

## Step 6: Load Sample People Data

Using MERGE for idempotent writes - safe to re-run.

In [ ]:
people = [
    {"email": "alice@org.com", "name": "Alice Chen", "title": "Senior Platform Engineer", "team": "Platform"},
    {"email": "bob@org.com", "name": "Bob Kumar", "title": "Staff ML Engineer", "team": "AI/ML"},
    {"email": "carol@org.com", "name": "Carol Diaz", "title": "Principal Architect", "team": "Architecture"},
    {"email": "dave@org.com", "name": "Dave Park", "title": "Senior DevOps Engineer", "team": "Platform"},
    {"email": "eve@org.com", "name": "Eve Johnson", "title": "Security Engineer", "team": "Security"},
]

for person in people:
    query = """
    MERGE (p:Person {email: $email})
    SET p.name = $name, p.title = $title, p.team = $team,
        p.expertise_score = 0.0, p.pagerank = 0.0
    RETURN p.name AS name
    """
    result = run_query(query, parameters={
        "email": {"value": person["email"]},
        "name": {"value": person["name"]},
        "title": {"value": person["title"]},
        "team": {"value": person["team"]}
    })
    print(f"  \u2713 Merged person: {person['name']}")

print(f"\u2713 {len(people)} people loaded")

## Step 7: Load Topics and Relationships

In [ ]:
topics = [
    {"name": "Kubernetes", "category": "infrastructure"},
    {"name": "Authentication", "category": "security"},
    {"name": "Machine Learning", "category": "ai"},
    {"name": "Graph Databases", "category": "data"},
    {"name": "CI/CD", "category": "devops"},
    {"name": "Microservices", "category": "architecture"},
]

for topic in topics:
    query = """
    MERGE (t:Topic {name: $name})
    SET t.category = $category
    RETURN t.name AS name
    """
    run_query(query, parameters={
        "name": {"value": topic["name"]},
        "category": {"value": topic["category"]}
    })

print(f"\u2713 {len(topics)} topics loaded")

In [ ]:
# Create KNOWS_ABOUT relationships with strength scores
expertise_links = [
    ("alice@org.com", "Kubernetes", 0.9),
    ("alice@org.com", "CI/CD", 0.8),
    ("bob@org.com", "Machine Learning", 0.95),
    ("bob@org.com", "Graph Databases", 0.7),
    ("carol@org.com", "Microservices", 0.9),
    ("carol@org.com", "Authentication", 0.75),
    ("dave@org.com", "Kubernetes", 0.85),
    ("dave@org.com", "CI/CD", 0.9),
    ("eve@org.com", "Authentication", 0.95),
    ("eve@org.com", "Microservices", 0.6),
]

for email, topic_name, strength in expertise_links:
    query = """
    MATCH (p:Person {email: $email})
    MATCH (t:Topic {name: $topic})
    MERGE (p)-[r:KNOWS_ABOUT]->(t)
    SET r.strength = $strength
    RETURN p.name AS person, t.name AS topic
    """
    run_query(query, parameters={
        "email": {"value": email},
        "topic": {"value": topic_name},
        "strength": {"value": strength}
    })

print(f"\u2713 {len(expertise_links)} expertise relationships created")

## Step 8: Load Sample Documents

In [ ]:
documents = [
    {"doc_id": "doc-001", "title": "K8s Migration Playbook", "author": "alice@org.com", "topic": "Kubernetes"},
    {"doc_id": "doc-002", "title": "ML Pipeline Architecture", "author": "bob@org.com", "topic": "Machine Learning"},
    {"doc_id": "doc-003", "title": "Zero Trust Auth Design", "author": "eve@org.com", "topic": "Authentication"},
    {"doc_id": "doc-004", "title": "Service Mesh Patterns", "author": "carol@org.com", "topic": "Microservices"},
    {"doc_id": "doc-005", "title": "GitOps Pipeline Setup", "author": "dave@org.com", "topic": "CI/CD"},
]

for doc in documents:
    query = """
    MERGE (d:Document {doc_id: $doc_id})
    SET d.title = $title
    WITH d
    MATCH (p:Person {email: $author})
    MERGE (p)-[:AUTHORED]->(d)
    WITH d
    MATCH (t:Topic {name: $topic})
    MERGE (d)-[:RELATED_TO]->(t)
    RETURN d.title AS title
    """
    run_query(query, parameters={
        "doc_id": {"value": doc["doc_id"]},
        "title": {"value": doc["title"]},
        "author": {"value": doc["author"]},
        "topic": {"value": doc["topic"]}
    })

print(f"\u2713 {len(documents)} documents with authorship and topic links")

## Step 9: Verify Graph Schema

In [ ]:
# Count nodes by label
count_query = """
MATCH (n)
RETURN labels(n)[0] AS label, count(n) AS count
ORDER BY count DESC
"""
result = run_query(count_query)
print("Node counts:")
for row in result["results"]:
    print(f"  {row['label']}: {row['count']}")

# Count relationships
rel_query = """
MATCH ()-[r]->()
RETURN type(r) AS relationship, count(r) AS count
ORDER BY count DESC
"""
result = run_query(rel_query)
print("\nRelationship counts:")
for row in result["results"]:
    print(f"  {row['relationship']}: {row['count']}")

print("\n\u2713 Graph schema verified")

## Step 10: Sample Query - Find Experts

In [ ]:
# Find top experts for a given topic
expert_query = """
MATCH (p:Person)-[r:KNOWS_ABOUT]->(t:Topic {name: $topic})
RETURN p.name AS expert, p.title AS title, r.strength AS strength
ORDER BY r.strength DESC
LIMIT 5
"""
result = run_query(expert_query, parameters={"topic": {"value": "Kubernetes"}})
print("Top Kubernetes experts:")
for row in result["results"]:
    print(f"  {row['expert']} ({row['title']}) - strength: {row['strength']}")

print("\n\u2713 Expert query working")

## Step 11: Save Configuration

In [ ]:
config["NEPTUNE_GRAPH_ID"] = graph_id
config["NEPTUNE_ENDPOINT"] = endpoint
Path("config.json").write_text(json.dumps(config, indent=2))

print(f"\u2713 Config saved")
print(f"  NEPTUNE_GRAPH_ID: {graph_id}")
print(f"  NEPTUNE_ENDPOINT: {endpoint}")

## Summary

- [x] Neptune Analytics graph created with vector search (1024d)
- [x] Schema indexes defined (Person, Document, Topic, Team)
- [x] Sample data loaded (5 people, 6 topics, 5 documents)
- [x] Expertise relationships established (KNOWS_ABOUT with strength)
- [x] Expert query validated
- [x] config.json updated with graph ID and endpoint

**Next:** [03-embeddings.ipynb](03-embeddings.ipynb) — Generate Titan V2 embeddings and store on Neptune for similarity search